## Appendix 1 — Quadrant Based Concentration Criterion for Filtering Out Sparse Tiles

In **Section 5.2** we describe how each WSI is broken into small patches (“tiles”), then filtered so that tiles containing little or no tissue are discarded. Below is a more detailed explanation of how we decide whether a tile is classified as **background** (low signal) or **valid** (high tissue content), drawing directly on our tiler.py script logic:

To ensure that each tile contains sufficient tissue (i.e., non-background pixels), we employ a **two-stage** filtering process:

1. **Global Signal Fraction**  
   - Convert the tile to grayscale and create a boolean mask `signal_mask`:
     ```text
     signal_mask[x,y] = 1  if grayscale[x,y] < 240,
                       = 0  otherwise.
     ```
   - The global signal fraction is then:
     $$
       \text{signal_fraction}
       = \frac{\sum_{(x,y)} \text{signal_mask}[x,y]}{\text{(number of pixels in tile)}}
       .
     $$

     If $\text{signal_fraction} < 0.9$ (for example), the tile is discarded as too background-heavy.



2. **Quadrant Concentration Check**  
   - For tiles passing the first test, subdivide the tile into four quadrants:
     $$       Q_{\text{TL}},\; Q_{\text{TR}},\; Q_{\text{BL}},\; Q_{\text{BR}}. $$
   - For each quadrant $Q$, compute
     $$\text{q_fraction}(Q)= \frac{\text{(sum of signal pixels in }Q)}{\text{(number of pixels in }Q)}.$$
   
   - Then define
     $$
       \max Q
       = \max\!\bigl\{
         \text{q_fraction}(Q_{\text{TL}}),\,
         \text{q_fraction}(Q_{\text{TR}}),\,
         \text{q_fraction}(Q_{\text{BL}}),\,
         \text{q_fraction}(Q_{\text{BR}})
       \bigr\},
     $$
     
     and let
     $$
       \text{concentration}
       = \frac{\max Q}{\text{signal_fraction} + 10^{-6}}
       .
     $$
     
        - If signal is concentrated, i,e, If $\text{concentration} \ge 0.7$ (for example)- then we discard the tile since the fact that the signal is concentrated (high fraction in some quadrants, low in others) suggests it's a **border tile**



---

**Example Pseudocode**:

    for each tile in WSI:
    tile_gray = convert_to_grayscale(tile)
    signal_mask = (tile_gray < 240)

    signal_fraction = sum(signal_mask) / total_pixels(tile_gray)
    if signal_fraction < tissue_threshold:
        discard tile
    else:
        # Subdivide into quadrants
        quadrants = [top_left, top_right, bottom_left, bottom_right]
        q_fractions = [ sum(q) / size(q) for q in quadrants ]
        max_quadrant = max(q_fractions)

        concentration = max_quadrant / (signal_fraction + 1e-6)
        if concentration >= concentration_threshold:
            discard tile
        else:
            keep tile

Using this approach, we retain only tiles with a sufficiently high **signal fraction** *and* no high quadrant concentration tiles, helping to discard background- or border tiles.